# 02 · rent.com.au scraping, split between 5 people (Sept 2026 snapshot)

This notebook **plans, checks and merges** the group's scraping of rent.com.au. The scraping itself runs in each
person's terminal with `scripts/scrape_rentcomau.py`. It can be stopped and restarted at any time, and it carries
on where it left off. Nothing in this notebook sends requests to rent.com.au.

**How the work is split.** The 654 suburbs that had listings in the 2025 course dataset are dealt out like cards
(biggest first, in a 1-2-3-4-5-5-4-3-2-1 order), so each person gets about 131 suburbs and the same amount of work.
The split is built the same way on every computer, so nobody scrapes a suburb twice. The script refuses any share
that isn't out of 5 (`--shard 2/5`), so shares can't overlap.

**Steps (the step-by-step guide for each person is in the group chat)**

1. Everyone: `git fetch`, `git switch kerri-updates`, and put the course Domain folder in `data/raw/domain/Data/`.
2. Everyone: one quick check (2 requests): `python3 scripts/scrape_rentcomau.py test --contact <your uni email>`
3. Everyone: run your **search** command from section 1 (about 20–40 minutes).
4. Everyone: zip your `search/scraperN` folder and upload it to the group's Google Drive folder.
5. Everyone: start your **details** command and leave it running overnight. Upload `details/scraperN.csv` when it finishes.
6. Coordinator: download every zip, unzip into the repo's top folder, then run sections 3 and 4 below.
7. Coordinator: build the master table (2025 + 2026) at the end of `01_data_curation`, then re-run the later notebooks.

**Scraping politely (built into the script).** It reads robots.txt and skips anything disallowed. It waits about
5 seconds between requests and identifies itself as a University of Melbourne student project with your email.
**It stops immediately if the site refuses access** and does not try to get around it; if that happens, tell
the group. It never saves agents' or landlords' names or phone numbers. Scraped files are never committed to
GitHub (the repo is public), only shared through the Drive folder.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import pandas as pd
import config
import scrape_rentcomau as scr

SNAPSHOT = scr.DEFAULT_SNAPSHOT                                     # "2026-09", the same for everyone
TEAM = ["scraper1", "scraper2", "scraper3", "scraper4", "scraper5"]  # placeholders: share 1 to 5, in order
assert len(TEAM) == scr.GROUP_SIZE, "TEAM must have one name per share"
print(f"Snapshot {SNAPSHOT}, {scr.GROUP_SIZE} people")

Snapshot 2026-09, 5 people


## 1. Who scrapes which suburbs

In [2]:
plan = scr.make_plan(scr.load_suburbs(), len(TEAM))
plan["worker"] = plan["shard"].map(dict(enumerate(TEAM, start=1)))
out = scr.snap_dir(SNAPSHOT)
out.mkdir(parents=True, exist_ok=True)
plan.to_csv(out / "plan.csv", index=False)

shares = plan.groupby(["shard", "worker"]).agg(
    suburbs=("suburb", "size"), est_pages=("est_pages", "sum"),
    domain_2025_listings=("listing_count", "sum"),
    biggest_suburbs=("suburb", lambda s: ", ".join(s.head(5).str.title())))
shares["est_search_minutes"] = (shares["est_pages"] * 5 / 60).round(0).astype(int)
shares

,,suburbs,est_pages,domain_2025_listings,biggest_suburbs,est_search_minutes
shard,worker,,,,,
1,scraper1,131,196,2577,"Melbourne, Footscray, Clayton, Reservoir, Sout...",16
2,scraper2,131,195,2571,"Southbank, Docklands, Craigieburn, Shepparton,...",16
3,scraper3,131,194,2560,"Tarneit, St Kilda, Carlton, Box Hill, Wollert",16
4,scraper4,131,193,2509,"Truganina, Werribee, Pakenham, Berwick, Melton...",16
5,scraper5,130,191,2498,"South Yarra, Point Cook, Wyndham Vale, North M...",16


The page estimates use the 2025 Domain counts; rent.com.au often shows more listings per suburb (Box Hill: 270
vs 111), so allow about 1.5× (20–40 minutes). Each person can print their full suburb list with
`python3 scripts/scrape_rentcomau.py plan --shard N/5`. Each person's commands:

In [3]:
for shard, name in enumerate(TEAM, start=1):
    print(f"{name} (share {shard}/{len(TEAM)}):")
    print(f"  python3 scripts/scrape_rentcomau.py search --shard {shard}/{len(TEAM)} --worker {name} --contact YOUR_UNI_EMAIL")
    print(f"  python3 scripts/scrape_rentcomau.py details --worker {name} --contact YOUR_UNI_EMAIL\n")

scraper1 (share 1/5):
  python3 scripts/scrape_rentcomau.py search --shard 1/5 --worker scraper1 --contact YOUR_UNI_EMAIL
  python3 scripts/scrape_rentcomau.py details --worker scraper1 --contact YOUR_UNI_EMAIL

scraper2 (share 2/5):
  python3 scripts/scrape_rentcomau.py search --shard 2/5 --worker scraper2 --contact YOUR_UNI_EMAIL
  python3 scripts/scrape_rentcomau.py details --worker scraper2 --contact YOUR_UNI_EMAIL

scraper3 (share 3/5):
  python3 scripts/scrape_rentcomau.py search --shard 3/5 --worker scraper3 --contact YOUR_UNI_EMAIL
  python3 scripts/scrape_rentcomau.py details --worker scraper3 --contact YOUR_UNI_EMAIL

scraper4 (share 4/5):
  python3 scripts/scrape_rentcomau.py search --shard 4/5 --worker scraper4 --contact YOUR_UNI_EMAIL
  python3 scripts/scrape_rentcomau.py details --worker scraper4 --contact YOUR_UNI_EMAIL

scraper5 (share 5/5):
  python3 scripts/scrape_rentcomau.py search --shard 5/5 --worker scraper5 --contact YOUR_UNI_EMAIL
  python3 scripts/scrape_rentc

## What gets scraped

**Search stage** (one file per suburb: `data/raw/rentcomau/2026-09/search/<name>/<Suburb>_<postcode>.csv`)

| Column | Meaning |
|---|---|
| `listing_id`, `url` | rent.com.au's id for the listing and its page |
| `address`, `suburb`, `postcode` | where the property is |
| `weekly_rent`, `price_text` | rent in $ per week, and the price exactly as shown |
| `bedrooms`, `bathrooms`, `carspaces` | room counts |
| `property_type` | House, Apartment, Unit, Townhouse, Studio, Other… |
| `lat`, `lon` | coordinates (used for SA2, distances and maps) |
| `date_listed`, `available_date` | when the ad went up, and when the property can be moved into |
| `pets_allowed`, `agency` | pets yes/no, and the agency's business name |
| `walk_score`, `transit_score` | 0–100 scores shown by rent.com.au (not in the 2025 data) |
| `search_suburb`, `search_postcode`, `in_searched_suburb`, `page` | which suburb search found it, whether it is really in that suburb, which results page |
| `parse_method`, `worker`, `scraped_date` | how it was read, who scraped it, when |

**Details stage** (`data/raw/rentcomau/2026-09/details/<name>.csv`, one row per listing)

| Column | Meaning |
|---|---|
| `bond` | bond in $ |
| `structured_features` | listed features, e.g. "Heating: gas, Kitchen: dishwasher" |
| `furnished`, `partially_furnished`, `ensuites` | furnishing and number of ensuites |
| `photo_count`, `byline`, `description` | number of photos, headline, ad text (phone numbers and emails removed) |
| `weekly_rent_detail`, `detail_scraped_date` | rent as shown on the listing page, and when it was read |

**Never saved:** agents' and landlords' names, phone numbers and emails.

**After merging** (section 4), everything is combined into `vic_rentals_rentcomau.csv`, whose first 20 columns have
the same names and order as the 2025 Domain file (`listing_id, suburb, postcode, weekly_rent, bond, available_date,
date_listed, days_listed, bedrooms, bathrooms, carspaces, property_type, address, lat, lon, scraped_date,
photo_count, agency, structured_features, url`), followed by the rent.com.au extras.

## 2. Check the parser (no internet)

Parses the two pages saved by the last `test` run (`data/raw/rentcomau/debug/`). Only summaries are shown, because
the notebook is committed to a public repo.

In [4]:
try:
    rows, detail = scr.offline_check()
    parsed = pd.DataFrame(rows, columns=scr.SEARCH_COLUMNS)
    filled = parsed.notna().mean().mul(100).round(0).rename("% of listings with a value")
    display(filled[filled.index.isin(["weekly_rent", "bedrooms", "bathrooms", "carspaces", "property_type",
                                      "lat", "lon", "date_listed", "available_date", "agency",
                                      "walk_score", "transit_score"])].to_frame())
    if detail:
        print("Listing page fields found:", [k for k, v in detail.items() if v not in (None, "") and k != "listing_id"])
except SystemExit as e:
    print(e)

Saved search page: site says 307 listings; parsed 25 (method: page data)


,% of listings with a value
weekly_rent,100.0
bedrooms,100.0
bathrooms,100.0
carspaces,100.0
property_type,100.0
lat,100.0
lon,100.0
date_listed,100.0
available_date,100.0
agency,100.0


Listing page fields found: ['bond', 'weekly_rent_detail', 'furnished', 'partially_furnished', 'ensuites', 'structured_features', 'byline', 'description', 'photo_count', 'detail_scraped_date']


## 3. Progress

Counts the suburb files in `data/raw/rentcomau/2026-09/search/<name>/`. Each person can check their own progress with
`python3 scripts/scrape_rentcomau.py status`; here the coordinator sees everyone's once the zips are unzipped.

In [5]:
table, details = scr.status_table(SNAPSHOT, len(TEAM))
table.index = [f"{s} ({TEAM[s - 1]})" for s in table.index]
print("Listing-page details collected:", details or "none yet")
table

Listing-page details collected: {'scraper4': 1587, 'scraper5': 1644, 'scraper3': 1524, 'scraper2': 1684, 'scraper1': 1752}


,suburbs,done,listings_found,workers,percent_done
1 (scraper1),131,131,13618,scraper1,100.0
2 (scraper2),131,131,12953,scraper2,100.0
3 (scraper3),131,131,13326,scraper3,100.0
4 (scraper4),131,131,14192,scraper4,100.0
5 (scraper5),130,130,13533,scraper5,100.0


## 4. Merge everyone's files and check them

Before running this: download each person's zip from the Drive folder and unzip it **in the repo's top folder**, so
the files land in `data/raw/rentcomau/2026-09/search/<name>/` (and `details/<name>.csv`).

The merge combines every person's files into `data/raw/rentcomau/2026-09/vic_rentals_rentcomau.csv`, the file that
`scripts/listings.py` reads as the 2026 snapshot. A listing can appear in more than one suburb's search (and so in
two people's files); one copy is kept, preferring the search of its own suburb. Re-run this cell whenever more files
arrive (for example the details files the next morning).

In [6]:
merged = scr.merge(SNAPSHOT)
if merged is not None:
    domain_cols = pd.read_csv(config.DOMAIN_2025, nrows=0).columns
    print("\nDomain columns we also have:", sorted(set(domain_cols) & set(merged.columns)))
    print("Domain columns rent.com.au doesn't show:", sorted(set(domain_cols) - set(merged.columns)))
    display(merged.groupby("worker").agg(listings=("listing_id", "size"), suburbs=("search_suburb", "nunique"),
                                         median_weekly_rent=("weekly_rent", "median"),
                                         with_details=("detail_scraped_date", lambda s: s.notna().sum())))

/home/dharmawan/industry-project-group-55/scripts/scrape_rentcomau.py:764: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  search = pd.concat([pd.read_csv(f, dtype={"listing_id": str}) for f in files], ignore_index=True)


Merged 654 suburb files from 5 people: 67,622 rows -> 9,178 unique listings
  in their searched suburb: 8,191
  with weekly_rent           99.9%
  with lat                  100.0%
  with date_listed          100.0%
  with bond                  89.2%
  with structured_features   50.1%
Saved to data/raw/rentcomau/2026-09/vic_rentals_rentcomau.csv (read automatically by scripts/listings.py)

Domain columns we also have: ['address', 'agency', 'available_date', 'bathrooms', 'bedrooms', 'bond', 'carspaces', 'date_listed', 'days_listed', 'lat', 'listing_id', 'lon', 'photo_count', 'postcode', 'property_type', 'scraped_date', 'structured_features', 'suburb', 'url', 'weekly_rent']
Domain columns rent.com.au doesn't show: ['agency_id', 'agent_names', 'domain_page_id', 'floorplans_count', 'land_area', 'primary_type', 'property_id', 'secondary_type', 'video_count', 'virtual_tour']


,listings,suburbs,median_weekly_rent,with_details
worker,,,,
scraper1,1814,97,600.0,1752
scraper2,1973,108,600.0,1684
scraper3,1720,103,585.0,1524
scraper4,1755,104,545.0,1587
scraper5,1916,105,550.0,1644


## 5. Next

Build the master table (2025 + 2026 stacked, one set of cleaning rules) at the end of `01_data_curation`, then
run the later notebooks (03 to 06). Every map and table gains a 2026 version automatically.